# HUP164: Compare Three Ictal Seizure Runs

This notebook repeats the same analysis for HUP164 ictal `run-01`, `run-02`, and `run-03`.

It will:

1. Load metadata for each run
2. Read `events.tsv` for seizure onset/offset
3. Read `channels.tsv` for bad channels and SOZ channels
4. Remove bad channels
5. Plot SOZ channels around seizure onset
6. Extract the same features for all runs
7. Compare SOZ vs non-SOZ across all runs
8. Compare preictal candidate vs early ictal changes
9. Show top abnormal channels by feature increase

**Note:** This notebook uses normal `tqdm` instead of `tqdm.notebook`, so it should not give the `IProgress not found` error.


In [ ]:
# =========================
# Cell 1: Imports and paths
# =========================

from pathlib import Path
import json
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

from scipy.signal import welch
from tqdm import tqdm  # safer than tqdm.notebook; avoids IProgress / ipywidgets error

mne.set_log_level("WARNING")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 120)

DATA_DIR = Path(r"G:\SAAD\medical\HUP_iEEG_Project\data\HUP_iEEG_Epilepsy_Dataset")

SUBJECT = "sub-HUP164"
SESSION = "ses-presurgery"
IEEG_DIR = DATA_DIR / SUBJECT / SESSION / "ieeg"

print("Data folder exists:", DATA_DIR.exists())
print("Subject folder exists:", (DATA_DIR / SUBJECT).exists())
print("iEEG folder exists:", IEEG_DIR.exists())
print("iEEG folder:", IEEG_DIR)


In [ ]:
# =====================================
# Cell 2: Define the three ictal runs
# =====================================

runs = ["01", "02", "03"]

run_files = {}

for run in runs:
    base = f"{SUBJECT}_{SESSION}_task-ictal_acq-seeg_run-{run}"
    run_files[run] = {
        "base": base,
        "edf": IEEG_DIR / f"{base}_ieeg.edf",
        "channels": IEEG_DIR / f"{base}_channels.tsv",
        "events": IEEG_DIR / f"{base}_events.tsv",
        "json": IEEG_DIR / f"{base}_ieeg.json",
    }

for run, files in run_files.items():
    print(f"
Run {run}")
    for key, path in files.items():
        if key != "base":
            print(f"{key:10s}", path.exists(), path.name)


## Step 1: Read metadata, events, and channels

For each run, this section reads:

- `channels.tsv`: channel names, bad channels, SOZ channels
- `events.tsv`: seizure onset and offset times
- `ieeg.json`: recording metadata
- `.edf` header: sampling frequency, duration, channel names


In [ ]:
# ===========================================================
# Cell 3: Helper function to read metadata for one ictal run
# ===========================================================

def read_run_metadata(run, files):
    channels_df = pd.read_csv(files["channels"], sep="	")
    events_df = pd.read_csv(files["events"], sep="	")

    with open(files["json"], "r") as f:
        metadata = json.load(f)

    raw_header = mne.io.read_raw_edf(files["edf"], preload=False, verbose=False)

    onset_rows = events_df[
        events_df["trial_type"].astype(str).str.lower().str.contains("onset", na=False)
    ]
    offset_rows = events_df[
        events_df["trial_type"].astype(str).str.lower().str.contains("offset", na=False)
    ]

    if len(onset_rows) == 0 or len(offset_rows) == 0:
        raise ValueError(f"Could not find seizure onset/offset in events file for run-{run}")

    seizure_onset = float(onset_rows["onset"].iloc[0])
    seizure_offset = float(offset_rows["onset"].iloc[0])

    bad_channels = channels_df.loc[
        channels_df["status"].astype(str).str.lower() == "bad",
        "name"
    ].tolist()

    soz_channels = channels_df.loc[
        channels_df["status_description"].astype(str).str.lower().str.contains("soz", na=False),
        "name"
    ].tolist()

    info = {
        "run": run,
        "edf_file": files["edf"].name,
        "n_channels_channels_tsv": channels_df.shape[0],
        "n_channels_edf": len(raw_header.ch_names),
        "sfreq": raw_header.info["sfreq"],
        "recording_duration_sec": float(raw_header.times[-1]),
        "recording_duration_min": float(raw_header.times[-1] / 60),
        "seizure_onset_sec": seizure_onset,
        "seizure_offset_sec": seizure_offset,
        "seizure_duration_sec": seizure_offset - seizure_onset,
        "time_before_seizure_sec": seizure_onset,
        "time_after_seizure_sec": float(raw_header.times[-1] - seizure_offset),
        "n_bad_channels": len(bad_channels),
        "bad_channels": bad_channels,
        "n_soz_channels": len(soz_channels),
        "soz_channels": soz_channels,
        "channels_df": channels_df,
        "events_df": events_df,
        "metadata": metadata,
        "raw_ch_names": raw_header.ch_names,
    }

    return info


In [ ]:
# ===============================================
# Cell 4: Load metadata for run-01, run-02, run-03
# ===============================================

all_run_info = {}

for run in runs:
    print(f"Reading metadata for run-{run}...")
    all_run_info[run] = read_run_metadata(run, run_files[run])

print("Done.")


In [ ]:
# =====================================
# Cell 5: Basic run comparison table
# =====================================

run_summary_rows = []

for run, info in all_run_info.items():
    run_summary_rows.append({
        "run": f"run-{run}",
        "edf_file": info["edf_file"],
        "n_channels_edf": info["n_channels_edf"],
        "sfreq": info["sfreq"],
        "recording_duration_sec": info["recording_duration_sec"],
        "recording_duration_min": info["recording_duration_min"],
        "seizure_onset_sec": info["seizure_onset_sec"],
        "seizure_offset_sec": info["seizure_offset_sec"],
        "seizure_duration_sec": info["seizure_duration_sec"],
        "time_before_seizure_sec": info["time_before_seizure_sec"],
        "time_after_seizure_sec": info["time_after_seizure_sec"],
        "n_bad_channels": info["n_bad_channels"],
        "n_soz_channels": info["n_soz_channels"],
    })

run_info_df = pd.DataFrame(run_summary_rows)
display(run_info_df)


In [ ]:
# ===============================================
# Cell 6: Display events table for each ictal run
# ===============================================

for run, info in all_run_info.items():
    print(f"
================ RUN {run} EVENTS ================")
    display(info["events_df"])


In [ ]:
# ==================================================
# Cell 7: Display bad channels and SOZ channels
# ==================================================

for run, info in all_run_info.items():
    print(f"
================ RUN {run} CHANNEL LABELS ================")
    print("Bad channels:")
    print(info["bad_channels"])

    print("
SOZ channels:")
    print(info["soz_channels"])


In [ ]:
# ====================================================
# Cell 8: Check SOZ and bad channel consistency
# ====================================================

soz_sets = {run: set(info["soz_channels"]) for run, info in all_run_info.items()}
bad_sets = {run: set(info["bad_channels"]) for run, info in all_run_info.items()}

common_soz = set.intersection(*soz_sets.values())
all_soz = set.union(*soz_sets.values())

common_bad = set.intersection(*bad_sets.values())
all_bad = set.union(*bad_sets.values())

print("Common SOZ channels in all runs:")
print(sorted(common_soz))

print("
All SOZ channels found across runs:")
print(sorted(all_soz))

print("
Common bad channels in all runs:")
print(sorted(common_bad))

print("
All bad channels found across runs:")
print(sorted(all_bad))


## Step 2: Visualize SOZ channels around seizure onset

The seizure onset time may be different in each run. Therefore, these plots use **relative time**:

- `0 sec` = seizure onset
- negative time = before seizure onset
- positive time = after seizure onset


In [ ]:
# ===============================================
# Cell 9: Function to load one run and remove bad channels
# ===============================================

def load_clean_raw(run):
    files = run_files[run]
    info = all_run_info[run]

    raw = mne.io.read_raw_edf(files["edf"], preload=True, verbose=False)

    bad_channels_in_raw = [
        ch for ch in info["bad_channels"]
        if ch in raw.ch_names
    ]

    raw.info["bads"] = bad_channels_in_raw
    raw.pick(picks="data", exclude="bads")

    return raw


In [ ]:
# =====================================================
# Cell 10: Function to plot SOZ channels around onset
# =====================================================

def plot_soz_around_onset(run, seconds_before=20, seconds_after=20):
    info = all_run_info[run]
    onset = info["seizure_onset_sec"]

    raw = load_clean_raw(run)
    sfreq = raw.info["sfreq"]

    soz_channels = [
        ch for ch in info["soz_channels"]
        if ch in raw.ch_names
    ]

    if len(soz_channels) == 0:
        print(f"No SOZ channels found in run-{run}")
        return

    start_sec = max(0, onset - seconds_before)
    stop_sec = min(raw.times[-1], onset + seconds_after)

    start_sample = int(start_sec * sfreq)
    stop_sample = int(stop_sec * sfreq)

    picks = [raw.ch_names.index(ch) for ch in soz_channels]

    data, times = raw[picks, start_sample:stop_sample]
    times_relative = times - onset

    plt.figure(figsize=(16, 8))

    offset = 0
    for i in range(data.shape[0]):
        signal = data[i] * 1e6
        signal = signal - np.mean(signal)
        plt.plot(times_relative, signal + offset, label=soz_channels[i])
        offset += np.std(signal) * 6

    plt.axvline(0, linestyle="--", linewidth=2, label="Seizure onset")
    plt.xlabel("Time relative to seizure onset (seconds)")
    plt.ylabel("SOZ channels with vertical offset")
    plt.title(f"HUP164 ictal run-{run}: SOZ channels around seizure onset")
    plt.legend(loc="upper right", fontsize=8)
    plt.tight_layout()
    plt.show()

    del raw
    gc.collect()


In [ ]:
# ===============================================
# Cell 11: Plot SOZ channels for all three runs
# ===============================================

for run in runs:
    plot_soz_around_onset(run, seconds_before=20, seconds_after=20)


In [ ]:
# ========================================================
# Cell 12: Plot one SOZ channel in three time segments
# ========================================================

def plot_single_channel_segments(run, channel_to_plot="LA1"):
    info = all_run_info[run]
    onset = info["seizure_onset_sec"]

    raw = load_clean_raw(run)
    sfreq = raw.info["sfreq"]

    if channel_to_plot not in raw.ch_names:
        print(f"{channel_to_plot} not found in run-{run}")
        del raw
        gc.collect()
        return

    ch_idx = raw.ch_names.index(channel_to_plot)

    segments = {
        "Before seizure: -20 to 0 sec": (onset - 20, onset),
        "Early seizure: 0 to +20 sec": (onset, onset + 20),
        "During seizure: +20 to +40 sec": (onset + 20, onset + 40),
    }

    for title, (start, stop) in segments.items():
        start = max(0, start)
        stop = min(raw.times[-1], stop)

        start_sample = int(start * sfreq)
        stop_sample = int(stop * sfreq)

        data, times = raw[ch_idx, start_sample:stop_sample]
        signal = data[0] * 1e6
        signal = signal - np.mean(signal)
        times_relative = times - onset

        plt.figure(figsize=(14, 4))
        plt.plot(times_relative, signal)
        plt.axvline(0, linestyle="--", linewidth=2)
        plt.xlabel("Time relative to seizure onset (seconds)")
        plt.ylabel("Amplitude (microvolts)")
        plt.title(f"run-{run} | {channel_to_plot} | {title}")
        plt.tight_layout()
        plt.show()

    del raw
    gc.collect()


In [ ]:
# ===============================================
# Cell 13: Plot LA1 for all runs
# ===============================================

for run in runs:
    plot_single_channel_segments(run, channel_to_plot="LA1")


## Step 3: Extract the same features from all runs

Features extracted for each channel and each region:

- `rms_uv`: average signal strength
- `peak_to_peak_uv`: maximum signal swing
- `std_uv`: signal variability
- `line_length_uv`: how rapidly/jaggedly the signal changes
- `delta_power`: 1–4 Hz
- `theta_power`: 4–8 Hz
- `alpha_power`: 8–13 Hz
- `beta_power`: 13–30 Hz
- `gamma_power`: 30–80 Hz
- `high_gamma_power`: 80–150 Hz


In [ ]:
# ==========================================
# Cell 14: Bandpower and feature functions
# ==========================================

def calculate_bandpower(signal, sfreq, low, high):
    freqs, psd = welch(
        signal,
        fs=sfreq,
        nperseg=min(len(signal), int(4 * sfreq))
    )
    mask = (freqs >= low) & (freqs <= high)
    return np.trapezoid(psd[mask], freqs[mask])


def create_regions_for_run(info):
    onset = info["seizure_onset_sec"]
    offset = info["seizure_offset_sec"]
    end = info["recording_duration_sec"]

    candidate_regions = [
        ["early_before", max(0, onset - 120), max(0, onset - 60), "Earlier pre-seizure baseline"],
        ["preictal_candidate", max(0, onset - 60), onset, "Last 60 seconds before seizure onset"],
        ["early_ictal", onset, min(offset, onset + 20), "First 20 seconds after seizure onset"],
        ["middle_ictal", min(offset, onset + 20), min(offset, onset + 40), "20 to 40 seconds after seizure onset"],
        ["late_ictal", min(offset, onset + 40), offset, "Later seizure period"],
        ["post_ictal", offset, end, "After seizure offset"],
    ]

    regions = []
    for region, start, stop, description in candidate_regions:
        if stop > start:
            regions.append([region, start, stop, description])

    return pd.DataFrame(regions, columns=["region", "start_sec", "end_sec", "description"])


def extract_features_for_region(raw_obj, channels, start_sec, end_sec, soz_channels):
    sfreq = raw_obj.info["sfreq"]

    start_sample = int(start_sec * sfreq)
    end_sample = int(end_sec * sfreq)

    picks = [raw_obj.ch_names.index(ch) for ch in channels if ch in raw_obj.ch_names]
    selected_channels = [raw_obj.ch_names[p] for p in picks]

    data, times = raw_obj[picks, start_sample:end_sample]

    data_uv = data * 1e6
    rows = []

    for i, ch in enumerate(selected_channels):
        x = data_uv[i]
        x = x - np.mean(x)

        rms = np.sqrt(np.mean(x ** 2))
        ptp = np.ptp(x)
        std = np.std(x)
        line_length = np.mean(np.abs(np.diff(x)))

        rows.append({
            "channel": ch,
            "is_soz": ch in soz_channels,
            "rms_uv": rms,
            "peak_to_peak_uv": ptp,
            "std_uv": std,
            "line_length_uv": line_length,
            "delta_power": calculate_bandpower(x, sfreq, 1, 4),
            "theta_power": calculate_bandpower(x, sfreq, 4, 8),
            "alpha_power": calculate_bandpower(x, sfreq, 8, 13),
            "beta_power": calculate_bandpower(x, sfreq, 13, 30),
            "gamma_power": calculate_bandpower(x, sfreq, 30, 80),
            "high_gamma_power": calculate_bandpower(x, sfreq, 80, 150),
            "start_sec": start_sec,
            "end_sec": end_sec,
        })

    return pd.DataFrame(rows)


In [ ]:
# ===============================================
# Cell 15: Create and show regions for each run
# ===============================================

all_regions = {}

for run in runs:
    regions_df = create_regions_for_run(all_run_info[run])
    all_regions[run] = regions_df

    print(f"
================ RUN {run} REGIONS ================")
    display(regions_df)


In [ ]:
# ========================================================
# Cell 16: Extract features for all three ictal runs
# ========================================================
# This cell uses normal tqdm, not tqdm.notebook, so it should not give IProgress error.

all_features = []

for run in runs:
    print(f"
Extracting features for run-{run}...")

    info = all_run_info[run]
    raw = load_clean_raw(run)

    good_channels = raw.ch_names
    soz_channels = info["soz_channels"]
    regions_df = all_regions[run]

    for _, region_row in tqdm(list(regions_df.iterrows()), total=len(regions_df)):
        region_features = extract_features_for_region(
            raw_obj=raw,
            channels=good_channels,
            start_sec=region_row["start_sec"],
            end_sec=region_row["end_sec"],
            soz_channels=soz_channels
        )

        region_features["run"] = f"run-{run}"
        region_features["region"] = region_row["region"]
        region_features["region_description"] = region_row["description"]

        all_features.append(region_features)

    del raw
    gc.collect()

features_all_runs_df = pd.concat(all_features, ignore_index=True)

display(features_all_runs_df.head())
print("Combined feature table shape:", features_all_runs_df.shape)


In [ ]:
# ===================================================
# Cell 17: Feature summary for SOZ vs non-SOZ
# ===================================================

feature_summary = features_all_runs_df.groupby(["run", "region", "is_soz"])[[
    "rms_uv",
    "peak_to_peak_uv",
    "std_uv",
    "line_length_uv",
    "delta_power",
    "theta_power",
    "alpha_power",
    "beta_power",
    "gamma_power",
    "high_gamma_power"
]].mean().reset_index()

display(feature_summary)


In [ ]:
# ===============================================
# Cell 18: Plot one feature across all runs
# ===============================================

def plot_metric_comparison(metric):
    plot_df = feature_summary.copy()

    plot_df["group"] = plot_df["run"] + " | " + plot_df["is_soz"].map({
        True: "SOZ",
        False: "Non-SOZ"
    })

    pivot_df = plot_df.pivot_table(
        index="region",
        columns="group",
        values=metric
    )

    region_order = [
        "early_before",
        "preictal_candidate",
        "early_ictal",
        "middle_ictal",
        "late_ictal",
        "post_ictal"
    ]

    existing_order = [r for r in region_order if r in pivot_df.index]
    pivot_df = pivot_df.loc[existing_order]

    display(pivot_df)

    pivot_df.plot(kind="bar", figsize=(16, 6))
    plt.ylabel(metric)
    plt.title(f"Comparison of {metric} across HUP164 ictal runs")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    return pivot_df


In [ ]:
# ===============================================
# Cell 19: Plot important feature comparisons
# ===============================================

rms_comparison = plot_metric_comparison("rms_uv")
line_length_comparison = plot_metric_comparison("line_length_uv")
gamma_comparison = plot_metric_comparison("gamma_power")
high_gamma_comparison = plot_metric_comparison("high_gamma_power")


In [ ]:
# ============================================================
# Cell 20: Fold change from preictal candidate to early ictal
# ============================================================

metrics_to_compare = [
    "rms_uv",
    "line_length_uv",
    "gamma_power",
    "high_gamma_power"
]

preictal_df = feature_summary[
    feature_summary["region"] == "preictal_candidate"
][["run", "is_soz"] + metrics_to_compare]

early_ictal_df = feature_summary[
    feature_summary["region"] == "early_ictal"
][["run", "is_soz"] + metrics_to_compare]

change_df = preictal_df.merge(
    early_ictal_df,
    on=["run", "is_soz"],
    suffixes=("_preictal", "_early_ictal")
)

for metric in metrics_to_compare:
    change_df[f"{metric}_fold_change"] = (
        change_df[f"{metric}_early_ictal"] /
        change_df[f"{metric}_preictal"]
    )

display(change_df)


In [ ]:
# ==========================================================
# Cell 21: Top abnormal channels by increase after onset
# ==========================================================

def get_channel_level_fold_change(features_df, metric):
    pre = features_df[
        features_df["region"] == "preictal_candidate"
    ][["run", "channel", "is_soz", metric]]

    early = features_df[
        features_df["region"] == "early_ictal"
    ][["run", "channel", "is_soz", metric]]

    merged = pre.merge(
        early,
        on=["run", "channel", "is_soz"],
        suffixes=("_preictal", "_early_ictal")
    )

    merged[f"{metric}_fold_change"] = (
        merged[f"{metric}_early_ictal"] /
        merged[f"{metric}_preictal"]
    )

    return merged

gamma_channel_change = get_channel_level_fold_change(features_all_runs_df, "gamma_power")
high_gamma_channel_change = get_channel_level_fold_change(features_all_runs_df, "high_gamma_power")
line_length_channel_change = get_channel_level_fold_change(features_all_runs_df, "line_length_uv")


In [ ]:
# =====================================================
# Cell 22: Show top channels for each run and metric
# =====================================================

def show_top_channels(change_table, metric, top_n=15):
    fold_col = f"{metric}_fold_change"

    for run in sorted(change_table["run"].unique()):
        print(f"
================ {run} TOP {top_n} CHANNELS BY {metric} INCREASE ================")

        top_df = change_table[
            change_table["run"] == run
        ].sort_values(fold_col, ascending=False).head(top_n)

        display(top_df[[
            "run",
            "channel",
            "is_soz",
            f"{metric}_preictal",
            f"{metric}_early_ictal",
            fold_col
        ]])

        n_soz_top = int(top_df["is_soz"].sum())
        print(f"SOZ channels in top {top_n}: {n_soz_top}/{top_n}")

show_top_channels(gamma_channel_change, "gamma_power", top_n=15)
show_top_channels(high_gamma_channel_change, "high_gamma_power", top_n=15)
show_top_channels(line_length_channel_change, "line_length_uv", top_n=15)


## Step 4: Interpretation guide

After running the notebook, use these questions to interpret the results:

1. Are seizure durations similar across run-01, run-02, and run-03?
2. Are SOZ channels the same across all three runs?
3. Do SOZ channels show stronger gamma/high-gamma increase after seizure onset?
4. Do non-SOZ channels become more active later, suggesting seizure spread?
5. Which feature is most consistent across runs: RMS, line length, gamma, or high-gamma?
6. In the top-channel tables, how many top channels are SOZ channels?

A good result would be that SOZ channels repeatedly show stronger early seizure changes across multiple ictal runs. That would make the analysis clinically meaningful.


In [ ]:
# ======================================================
# Cell 23: Quick automatic text summary from tables
# ======================================================

print("Basic run information:")
display(run_info_df)

print("
Fold change summary: preictal candidate -> early ictal")
display(change_df)

print("
Interpretation hints:")
for _, row in change_df.iterrows():
    group = "SOZ" if row["is_soz"] else "Non-SOZ"
    print(
        f"{row['run']} {group}: "
        f"gamma x{row['gamma_power_fold_change']:.2f}, "
        f"high-gamma x{row['high_gamma_power_fold_change']:.2f}, "
        f"line length x{row['line_length_uv_fold_change']:.2f}, "
        f"RMS x{row['rms_uv_fold_change']:.2f}"
    )
